# Reading JSON Data

## Creating a DataFrame using JSON Data

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
import json

spark = SparkSession.builder.appName("JsonExample").getOrCreate()

# Sample data with JSON strings
data = [("1", '{"name": "John", "age": 30}'), ("2", '{"name": "Alice", "age": 25}')]

# Creating DataFrame
df_json1 = spark.createDataFrame(data, ["id", "json_data"])

df_json1.show()

## Reading JSON data from a file and create a Dataframe out of it

In [0]:
df_json2 = spark.read.format('json').option('multiline','true').load('/Workspace/Users/sandipan.kar.data@gmail.com/Practice Dataset/JSON_Data_21.csv')
display(df_json2)

In [0]:
df_json2.printSchema()

# Different Schema Functions

## from_json

* Parses a JSON string column into a StructType.

In [0]:
display(df_json1)

In [0]:
from pyspark.sql.functions import col
display(df_json1.withColumn('schema_json_data', schema_of_json(col('json_data'))))

In [0]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import *

schema = StructType([
    StructField("name", StringType()),
    StructField("age", IntegerType()),
])

df_schemaed = df_json1.withColumn(
    "parsed_json",
    from_json("json_data", schema)
)
display(df_schemaed)

## to_json
* Converts a column containing complex data types like StructType, ArrayType, MapType, or VariantType into a JSON-formatted string column.

In [0]:
from pyspark.sql.functions import to_json
from pyspark.sql.types import *

df_json_text = df_schemaed.withColumn(
    "json_string",
    to_json("parsed_json")
)
display(df_json_text)

## get_json_object
* Extract single field

In [0]:
from pyspark.sql.functions import get_json_object

df_json1.select(
    get_json_object("json_data", "$.name")
).show()

df_schemaed.select(
    get_json_object("json_data", "$.name")
).show()

## json_tuple
* Extract multiple fields

In [0]:
from pyspark.sql.functions import json_tuple

df_json1.select(
    json_tuple("json_data", "age", "name")
).withColumnsRenamed(
    {"c0": "age", "c1": "name"}
).show()

## schema_of_json
* Infer JSON Schema

In [0]:
from pyspark.sql.functions import schema_of_json, col

display(
    df_json1.select(
    schema_of_json(
        col('json_data')
    )
))

## Parsing Complex JSON data using explode
* Flatten JSON array

In [0]:
df_json2.printSchema()

In [0]:
from pyspark.sql.functions import explode
df_json2_norm = df_json2.withColumn('batter_normalize', explode(col('batters.batter'))).withColumn('topping_normalize', explode(col('topping'))).drop('batters', 'topping')
df_json2_norm = df_json2_norm.select('id', 'name', 'ppu', 'type', col('batter_normalize.id').alias('batter_id'), col('batter_normalize.type').alias('batter_type'), col('topping_normalize.id').alias('topping_id'), col('topping_normalize.type').alias('topping_type'))
display(df_json2_norm)

#Dynamic JSON Implementation

In [0]:
complex_json = complex_json = """{
    "batch_id": "BATCH_001",
    "generated_at": "2026-02-15T10:30:00Z",
    "source": {
        "system": "CRM",
        "region": "APAC",
        "version": {
            "major": 2,
            "minor": 5
        }
    },
    "customers": [
        {
            "customer_id": "C001",
            "name": {
                "first": "Alice",
                "last": "Johnson"
            },
            "contact": {
                "emails": [
                    "alice@gmail.com",
                    "alice.work@company.com"
                ],
                "phones": [
                    {
                        "type": "mobile",
                        "number": "111-222-3333"
                    },
                    {
                        "type": "home",
                        "number": "444-555-6666"
                    }
                ]
            },
            "orders": [
                {
                    "order_id": "O1001",
                    "amount": 250.75,
                    "items": [
                        {
                            "product_id": "P01",
                            "qty": 2,
                            "price": 50
                        },
                        {
                            "product_id": "P02",
                            "qty": 1,
                            "price": 150
                        }
                    ],
                    "payments": [
                        {
                            "method": "card",
                            "status": "completed"
                        },
                        {
                            "method": "voucher",
                            "status": "applied"
                        }
                    ]
                },
                {
                    "order_id": "O1002",
                    "amount": 100,
                    "items": [],
                    "payments": null
                }
            ],
            "tags": [
                "premium",
                "newsletter_subscriber"
            ]
        },
        {
            "customer_id": "C002",
            "name": {
                "first": "Bob",
                "last": "Smith"
            },
            "contact": {
                "emails": [],
                "phones": [
                    {
                        "type": "mobile",
                        "number": "777-888-9999"
                    }
                ]
            },
            "orders": [
                {
                    "order_id": "O2001",
                    "amount": 500,
                    "items": [
                        {
                            "product_id": "P03",
                            "qty": 5,
                            "price": 100
                        }
                    ],
                    "payments": [
                        {
                            "method": "upi",
                            "status": "pending"
                        }
                    ]
                }
            ],
            "tags": null
        }
    ],
    "metadata": {
        "record_count": 2,
        "flags": {
            "test_data": false,
            "priority": "high"
        }
    }
}"""

dbutils.fs.put('dbfs:/Workspace/Users/sandipan.kar.data@gmail.com/Practice Dataset/Complex_JSON_Data_21.json', complex_json, overwrite=True)

In [0]:
df_json = spark.read.format('json').option('multiline', 'true').load('dbfs:/Workspace/Users/sandipan.kar.data@gmail.com/Practice Dataset/Complex_JSON_Data_21.json')

## Dynamic JSON Implementation V1

### Coding Implementation

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, ArrayType
import pandas as pd

class DynamicJsonParser:
    """
    Created By: Sandipan Kar
    Created On: 22/06/2026
    Description: 
    22/06: This class is used to parse a Spark DataFrame with nested JSON data and return a dictionary of metadata about the JSON structure.
    """

    def __init__(self, df):
        self.df = df
        self.df_hist = df
        self.schema = df.schema
        self.metadata = dict()
        self.metadata2 = None

    def json_parser(self, node=None, parent=None, lineage=[], lineage2 = ''):

        if node is None:
            node = self.schema
            lineage=[]

        # Root StructType
        if isinstance(node, StructType):

            for field in node.fields:
                #lineage2 = field.name
                #lineage.append(field.name) 
                self.json_parser(field, None, lineage, '')
                

        # StructField
        elif isinstance(node, StructField):            
            self.metadata[node.name] = [node.name, node.dataType.typeName(), parent, lineage, node.name]
            self.metadata2 = lineage2
            #print(f"Name: {node.name}, ", f"Type: {node.dataType.typeName()}", f"Parent: {parent}, ", f"Lineage: {lineage2}")

            # Struct
            if isinstance(node.dataType, StructType):
                lineage2 = node.name if lineage2 == '' else lineage2 + "." + node.name
                new_lineage = lineage + [node.name]
                for child in node.dataType.fields:
                    #lineage2 = lineage2 + "." + child.name
                    #lineage.append(child.name)
                    self.json_parser(child, node.name, new_lineage, lineage2)
                    

            # Array<Struct>
            elif (isinstance(node.dataType, ArrayType) and isinstance(node.dataType.elementType, StructType)):
                lineage2 = node.name if lineage2 == '' else lineage2 + "." + node.name
                new_lineage = lineage + [node.name]
                for child in node.dataType.elementType.fields:
                    #lineage2 = lineage2 + "." + child.name
                    #lineage.append(child.name)
                    self.json_parser(child, node.name, new_lineage, lineage2)
                    
        
    def json_formatter(self, drop = False, rename = False):
        self.json_parser()
        for key, value in self.metadata.items(): 
            if value[1] == 'array':
                col = key
                parent = value[2]
                lst = value[3]
                try:
                    lst.remove(parent) 
                except Exception as e:
                    pass
                str_lst = "_".join(lst)

                col_name = ''
                lineage = ''

                lineage = str_lst if str_lst != '' else lineage
                try:
                    if self.metadata[parent][1] == 'struct':
                        lineage = ((lineage + '.' if lineage != '' else lineage) + parent) if parent != '' and parent is not None else lineage
                    else:
                        lineage = ((lineage + '_' if lineage != '' else lineage) + parent) if parent != '' and parent is not None else lineage
                except Exception as e:
                    pass
                lineage = lineage + '.' + key if lineage != '' else key

                
                col_name = str_lst if str_lst != '' else col_name
                col_name = ((col_name + '_' if col_name != '' else col_name) + parent) if parent != '' and parent is not None else col_name
                col_name = col_name + '_' + key if col_name != '' else key
                
                
                self.metadata[key][4] = col_name
     
                self.df = self.df.withColumn(col_name, F.explode(lineage))
                
        for key, value in self.metadata.items(): 
            if value[1] == 'struct':
                col = key
                parent = value[2]
                lst = value[3]
                try:
                    lst.remove(parent) 
                except Exception as e:
                    pass
                str_lst = "_".join(lst)

                col_name = ''
                lineage = ''

                lineage = str_lst if str_lst != '' else lineage
                lineage = ((lineage + '.' if lineage != '' else lineage) + parent) if parent != '' and parent is not None else lineage
                lineage = lineage + '.' + key if lineage != '' else key

                
                col_name = str_lst if str_lst != '' else col_name
                col_name = ((col_name + '_' if col_name != '' else col_name) + parent) if parent != '' and parent is not None else col_name
                col_name = col_name + '_' + key if col_name != '' else key
                
                
                self.metadata[key][4] = col_name
   
                self.df = self.df.withColumn(col_name, F.col(lineage))
                
        for key, value in self.metadata.items(): 
            
            if value[1] != 'struct' and value[1] != 'array':

                col = key
                parent = value[2]
                lst = value[3]
                try:
                    lst.remove(parent) 
                except Exception as e:
                    pass
                str_lst = "_".join(lst)

                col_name = ''
                lineage = ''

                lineage = str_lst if str_lst != '' else lineage
                lineage = ((lineage + '.' if lineage != '' else lineage) + parent) if parent != '' and parent is not None else lineage
                lineage = lineage + '.' + key if lineage != '' else key

                
                col_name = str_lst if str_lst != '' else col_name
                col_name = ((col_name + '_' if col_name != '' else col_name) + parent) if parent != '' and parent is not None else col_name
                col_name = col_name + '_' + key if col_name != '' else key
                
                
                self.metadata[key][4] = col_name
                
                self.df = self.df.withColumn(col_name, F.col(lineage))
        
        """Drop and Rename Logic"""

        for key, value in self.metadata.items():
            if drop and (value[1] == 'array' or value[1] == 'struct'):
                self.df = self.df.drop(value[4])
            if rename and (value[1] != 'array' and value[1] != 'struct'):
                print(value[4]," ", value[0])
                self.df = self.df.withColumnRenamed(value[4], value[0])
            
        #display(self.df)

### Testing

In [0]:
cl = DynamicJsonParser(df_json)
cl.json_formatter(False, False)
df_json2 = cl.df
display(cl.df)